# Setup del cuaderno

In [ ]:
import types
import os 
import sys

def clean_globals():
    """
    Cleans the global environment by deleting variables.
    Preserves built-ins, imported modules, and this function itself.
    """
    for name in list(globals().keys()):
        # Do not delete built-in properties, imported modules, or the function itself
        if (
            not name.startswith('_') 
            and not isinstance(globals()[name], types.ModuleType)
            and name != 'clean_globals'  # <--- Crucial: prevents self-deletion
        ):
            del globals()[name]

## 1. Descriptivas de la base

In [ ]:
### load modules: descriptives
import types
import os 
import sys
import pandas as pd
import numpy as np

### 1.1 Importar datos

In [ ]:
### Cargar datos seleccionando las columnas requeridas
db = pd.read_csv("../stores/input/01_original_data_train.csv",
                 usecols=["pobre", "urbano", "numero_personas_hogar",
                          "cantidad_cuartos","arriendo","regimen_salud_2_household_average",
                          "edad_menor_18_household_average","oc_household_average",
                          "maximo_nivel_educativo_1_household_average",
                          "hh_female","hh_informal","hh_oc"])

### Transformar la variable dependiente
db["pobre"] = np.where(db["pobre"] == 'Yes', 1, 0)

### Computar informalidad condicional a estar trabajando y eliminar variable ocupado
db.loc[db['hh_oc'] == 0, 'hh_informal'] = np.nan
db = db.drop(columns=['hh_oc'])

### 1.2 Preparar tabla

In [ ]:
### Calcular tamaños de muestra (N) para los encabezados
n_pobre = db[db['pobre'] == 1].shape[0]
n_nopobre = db[db['pobre'] == 0].shape[0]

col_pobre = f"Pobre\n(N={n_pobre:,}, {n_pobre/(n_pobre+n_nopobre)*100:.2f}%)"
col_nopobre = f"No Pobre\n(N={n_nopobre:,}, {n_nopobre/(n_pobre+n_nopobre)*100:.2f}%))"

In [ ]:
rows = []

### Variables continuas: formato "Promedio (Desv Est)"
continuas = {
    'numero_personas_hogar': 'Personas por hogar',
    'cantidad_cuartos': 'Cantidad de cuartos',
    "arriendo": "Arriendo (COP)"
}

for col, nombre in continuas.items():
    mean_p = db.loc[db['pobre'] == 1, col].mean()
    std_p = db.loc[db['pobre'] == 1, col].std()
    
    mean_np = db.loc[db['pobre'] == 0, col].mean()
    std_np = db.loc[db['pobre'] == 0, col].std()
    
    # Ajuste para Arriendo (COP)
    if col == "arriendo":
        rows.append({
            'Variable': nombre,
            col_pobre: f"{mean_p:,.0f} ({std_p:,.0f})",
            col_nopobre: f"{mean_np:,.0f} ({std_np:,.0f})"
        })
    else:
        rows.append({
            'Variable': nombre,
            col_pobre: f"{mean_p:.2f} ({std_p:.2f})",
            col_nopobre: f"{mean_np:.2f} ({std_np:.2f})"
        })

# Variables categóricas/binarias: formato "XX.XX%"
categoricas = {
    'urbano': '¿El hgogar está ubicado zona urbana?',
    "regimen_salud_2_household_average":"Miembros del hogar en regimen subsidiado",
    "edad_menor_18_household_average":"Miembros del hogar menores a 18 años",
    "oc_household_average":"Miembros del hogar ocupados",
    "maximo_nivel_educativo_1_household_average": "Miembros del hogar educación primaria máximo",
    "hh_female": "¿Jefe del hogar es mujer? (%)",
    "hh_informal": "¿Jefe del hogar es un trabajador informal?(%)"
}

for col, nombre in categoricas.items():
    prop_p = db.loc[db['pobre'] == 1, col].mean() * 100
    prop_np = db.loc[db['pobre'] == 0, col].mean() * 100
    
    rows.append({
        'Variable': nombre,
        col_pobre: f"{prop_p:.2f}%",
        col_nopobre: f"{prop_np:.2f}%"
    })

In [ ]:
# Construir y mostrar el DataFrame final
tabla_descriptiva = pd.DataFrame(rows)
print(tabla_descriptiva)

### 1.3 Exportar

In [ ]:
tabla_descriptiva.to_latex('../stores/output/01_descriptives_main_document.tex',float_format="{:.2f}".format,escape=True,index=False)

## 2. Interpretabilidad

In [ ]:
### Clean enviroment
clean_globals()

In [ ]:
!pip install shap
!pip install rpy2 
!pip install xgboost

In [ ]:
### Load modules
import types
import os 
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import shap
import matplotlib.pyplot as plt
import operator
from xgboost import XGBClassifier

shap.initjs()

### 2.1 Importar datos y modelo

In [ ]:
### Cargar modelo
modelo = XGBClassifier()
modelo.load_model("../stores/input/01_original_r_trained_model_xgboost.ubj")

### Cargar datos
x_test = pd.read_csv('../stores/input/01_original_data_test.csv')

### Preparar datos
x_test = x_test.drop(columns=["id"])
x_test["pobre"] = np.where(x_test["pobre"] == "Yes", 1, 0)

y_test= x_test["pobre"]
x_test = x_test.drop(columns=["pobre"])

### 2.2 Computar SHAP

In [ ]:
X_sample = x_test
y_sample = y_test

explainer = shap.TreeExplainer(modelo)
shap_values_raw = explainer.shap_values(X_sample)

if isinstance(shap_values_raw, list):
    shap_values = shap_values_raw[1]
    base_value = explainer.expected_value[1]
elif shap_values_raw.ndim == 3:
    shap_values = shap_values_raw[:, :, 1]
    base_value = explainer.expected_value[1]
else:
    shap_values = shap_values_raw
    base_value = explainer.expected_value


In [ ]:
X_shap = x_test.copy()

X_shap = X_shap.rename(columns={
    "oc_household_average": "Ocupados (prom. hogar)",
    "arriendo": "Valor arriendo",
    "regimen_salud_1_household_average": "Régimen contributivo (prom. hogar)",
    "tipo_propiedad_vivienda_hogar": "Tipo de propiedad",
    "ocupado_horas_trabajadas_normalmente_household_working": "Horas trabajadas (prom. hogar)",
    "numero_personas_unidad_gasto": "Número de personas por unidad de gasto",
    "tipo_propiedad_vivienda_hogar_3": "Propiedad: en arriendo",
    "ocupado_tamano_de_la_empresa_9_household_working": "Empresa (101+ empleados)(prom. hogar)",
    "maximo_nivel_educativo_6_household_average": "Educación superior (prom. hogar)",
    "edad_menor_18_household_average": " Menores de 18 años (Prom. hogar)",
    "ocupado_relab_4_household_working": "Trabajadores por cuenta propia (Prom. hogar)"
})

### 2.3 Hacer gráfico

In [ ]:
plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values,
    X_shap,
    cmap="plasma",
    max_display=10,
    show=False
)

plt.tight_layout()
plt.savefig("../stores/output/02_shap.pdf", dpi=300, bbox_inches="tight")
plt.show()

## 3. Entrenar submodelo de 10 variables

In [ ]:
clean_globals()

In [ ]:
### load modules
import os 
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

### 3.1 Importar datos

In [ ]:
### Cargar datos de entrenamiento y validación
train = pd.read_csv('../stores/input/01_original_data_train.csv')
validation = pd.read_csv('../stores/input/01_original_data_validation.csv')
test = pd.read_csv('../stores/input/01_original_data_test.csv')

### Crear lista con 10 variables mas importantes del modelo grande (SHAP)
top_10_predictors = [
    "oc_household_average", "arriendo", "regimen_salud_1_household_average",
    "tipo_propiedad_vivienda_hogar", "ocupado_horas_trabajadas_normalmente_household_working",
    "numero_personas_unidad_gasto", "tipo_propiedad_vivienda_hogar_3",
    "ocupado_tamano_de_la_empresa_9_household_working", 
    "maximo_nivel_educativo_6_household_average", "ocupado_relab_4_household_working"
]

#### Limpiar variable dependiente
for df in [train, validation, test]:
    df["pobre"] = np.where(df["pobre"] == "Yes", 1, 0)

# Define X and y
train_x, train_y = train.drop(columns=["id", "pobre"]), train["pobre"]
validation_x, validation_y = validation.drop(columns=["id", "pobre"]), validation["pobre"]
test_x, test_y = test.drop(columns=["id", "pobre"]), test["pobre"]

### 3.2 Entrenar modelo

In [ ]:
### Entrenar el modelo con la misma arquitectura del modelo grande

model_top_10 = XGBClassifier(
    n_estimators=500, max_depth=7, learning_rate=0.04, gamma=0,
    colsample_bytree=0.6, subsample=0.8, min_child_weight=25,
    objective="binary:logistic", eval_metric="logloss", random_state=123
)

model_top_10.fit(train_x[top_10_predictors], train_y)

### 3.3 Exportar modelo

In [ ]:
### Guardar modelo
model_top_10.save_model('../stores/output/03_top_10.ubj')

## 4. Equidad y justicia algorítmica por grupo

In [ ]:
clean_globals()

In [ ]:
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

### 4.1 Importar modelo y datos

In [ ]:
### Cargar modelo original
modelo = XGBClassifier()
modelo.load_model("../stores/input/01_original_r_trained_model_xgboost.ubj")

### Cargar modelo top 10
model_top_10 = XGBClassifier()
model_top_10.load_model("../stores/output/03_top_10.ubj")

### Cargar datos
test = pd.read_csv('../stores/input/01_original_data_test.csv')
test_x, test_y = test.drop(columns=["id", "pobre"]), test["pobre"]
test["pobre"] = np.where(test["pobre"] == "Yes", 1, 0)

### Crear lista con 10 variables mas importantes del modelo grande (SHAP)
top_10_predictors = [
    "oc_household_average", "arriendo", "regimen_salud_1_household_average",
    "tipo_propiedad_vivienda_hogar", "ocupado_horas_trabajadas_normalmente_household_working",
    "numero_personas_unidad_gasto", "tipo_propiedad_vivienda_hogar_3",
    "ocupado_tamano_de_la_empresa_9_household_working", 
    "maximo_nivel_educativo_6_household_average", "ocupado_relab_4_household_working"]

### 4.2 Realizar predicciones

In [ ]:
### Fijar variables
base_cols = ['id', 'pobre', 'hh_female', 'urbano', 'hh_informal', 'hh_oc']

# Predictions: Main Model
test_main = test[base_cols].copy()
test_main['predicted_prob'] = modelo.predict_proba(test_x)[:, 1]
test_main['predicted_class'] = np.where(test_main['predicted_prob'] >= 0.323, 1, 0)

# Predictions: Top-10 Model
test_top10 = test[base_cols].copy()
test_top10['predicted_prob'] = model_top_10.predict_proba(test_x[top_10_predictors])[:, 1]
test_top10['predicted_class'] = np.where(test_top10['predicted_prob'] >= 0.34, 1, 0)

### 4.3 Calcular métricas y construir tablas

In [ ]:

### Definir una funcion para calcular las metricas por grupo
def calculate_fairness_metrics(df, group_col):
    results = []
    
    if group_col == 'hh_informal':
        df_eval = df[df['hh_oc'] == 1].copy()
    else:
        df_eval = df.copy()
        
    df_clean = df_eval.dropna(subset=[group_col])
    
    for group_val, group_df in df_clean.groupby(group_col, observed=False):
        tp = ((group_df['predicted_class'] == 1) & (group_df['pobre'] == 1)).sum()
        fn = ((group_df['predicted_class'] == 0) & (group_df['pobre'] == 1)).sum()
        fp = ((group_df['predicted_class'] == 1) & (group_df['pobre'] == 0)).sum()
        tn = ((group_df['predicted_class'] == 0) & (group_df['pobre'] == 0)).sum()
        
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        mean_prob = group_df['predicted_prob'].mean()
        
        results.append({
            'Variable': group_col,
            'Group': group_val,
            'TPR': tpr, 'FPR': fpr, 'Mean_Prob': mean_prob
        })
    return pd.DataFrame(results)

### Definir listado de variables sensibles
demographic_vars = ['hh_female', 'urbano', 'hh_informal']

### Aplicar funcion para el modelo agregado
metrics_main = pd.concat([calculate_fairness_metrics(test_main, var) for var in demographic_vars])
metrics_main.columns = ['Variable', 'Group', 'TPR_Main', 'FPR_Main', 'Mean_Prob_Main']

### Aplicar funcion para el modelo de 10 variables
metrics_top10 = pd.concat([calculate_fairness_metrics(test_top10, var) for var in demographic_vars])
metrics_top10.columns = ['Variable', 'Group', 'TPR_Top10', 'FPR_Top10', 'Mean_Prob_Top10']

### Unir dataframes
df_format = pd.merge(metrics_main, metrics_top10, on=['Variable', 'Group'], how='inner')

### 4.4 Preparar tabla

In [ ]:
### Renombrar los valores y las variables
nombres_vars = {
    'hh_female': '¿Jefe de hogar es \nde sexo femenino?', 
    'urbano': '¿El hogar reside en \nuna zona urbana?', 
    'hh_informal': '¿ El Jefe de hogar \nes un trabajador Informal? ^'
}
df_format['Variable'] = df_format['Variable'].replace(nombres_vars)
df_format['Group'] = df_format['Group'].apply(lambda x: 'Sí' if x == 1 else ('No' if x == 0 else x))

### Renombrar las metricas de la tabla
df_format = df_format.rename(columns={
    'Mean_Prob_Main': 'Demographic Parity_Main',
    'Mean_Prob_Top10': 'Demographic Parity_Top10'
})

### Aplicar estrellas de tolerancia
for model in ['Main', 'Top10']:
    for col in ['TPR', 'FPR', 'Demographic Parity']:
        col_name = f'{col}_{model}'
        
        # Calculate absolute difference between groups
        diff_series = df_format.groupby('Variable')[col_name].transform(lambda x: x.max() - x.min())
        
        def format_with_symbols(val, diff):
            if pd.isna(val) or pd.isna(diff): return f"{val:.4f}" if pd.notna(val) else str(val)
            if diff == 0: sym = "***"
            elif diff < 0.03: sym = "**"
            elif diff < 0.05: sym = "*"
            else: sym = ""
            return f"{val:.4f}{sym}"
        
        df_format[col_name] = df_format.apply(lambda row: format_with_symbols(row[col_name], diff_series[row.name]), axis=1)

### Aregar nueva linea para la visualización
df_format = df_format.set_index(['Variable', 'Group'])
df_format.index.names = ['Variable Sensible', 'Grupo']

### Crear columna multindice
column_tuples = [
    ('M: 152; (PC. 0,323)', 'TPR'), ('M: 152; (PC. 0,323)', 'FPR'), ('M: 152; (PC. 0,323)', 'Demographic Parity'),
    ('M: 10 ; (PC. 0,34)', 'TPR'), ('M: 10 ; (PC. 0,34)', 'FPR'), ('M: 10 ; (PC. 0,34)', 'Demographic Parity')
]
df_format.columns = pd.MultiIndex.from_tuples(column_tuples)

### Ver la tabla
df_format

### 4.5 Guardar tablas. 

In [ ]:
### Generar el código base con estilo
codigo_latex = df_format.style.to_latex(
    hrules=True,               
    multirow_align="t",        
    multicol_align="c"         
)

### Insertar manualmente las líneas divisorias para las métricas
cadena_buscar = r"\multicolumn{3}{c}{M: 10 ; (PC. 0,34)} \\"
reemplazo = r"\multicolumn{3}{c}{M: 10 ; (PC. 0,34)} \\" + "\n" + r"\cmidrule(lr){3-5} \cmidrule(lr){6-8}"

codigo_latex = codigo_latex.replace(cadena_buscar, reemplazo)

# Guardar
with open('../stores/output/04_tabla_equidad.tex', 'w', encoding='utf-8') as f:
    f.write(codigo_latex)

## 5. Mitigación por puntos de corte diferenciales: 

Para este punto usted debe instalar la librería error parity, la cual, necesita una versión de numphy anterior a la 2.0.0. 

In [ ]:
clean_globals()

In [ ]:
### Main Modules
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Models
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

### Metrics and calibrarion
from sklearn.calibration import calibration_curve
from error_parity import RelaxedThresholdOptimizer
from error_parity.pareto_curve import compute_postprocessing_curve
from error_parity.plotting import plot_postprocessing_solution
from sklearn.metrics import (f1_score,accuracy_score,recall_score,precision_score,roc_auc_score)

In [ ]:
import numpy as np
print("Versión de NumPy:", np.__version__)
print("Ruta del archivo:", np.__file__)

### 5.1 Cargar datos y modelo

In [ ]:
### Cargar modelo original
modelo = XGBClassifier()
modelo.load_model("../stores/input/01_original_r_trained_model_xgboost.ubj")

### Cargar modelo top 10
model_top_10 = XGBClassifier()
model_top_10.load_model("../stores/output/03_top_10.ubj")

### Cargar datos
test = pd.read_csv('../stores/input/01_original_data_test.csv')
validation = pd.read_csv('../stores/input/01_original_data_validation.csv')

### Limpiar dependiente
test["pobre"] = np.where(test["pobre"] == "Yes", 1, 0)
validation["pobre"] = np.where(validation["pobre"] == "Yes", 1, 0)

### Split datos
validation_x, validation_y = validation.drop(columns=["id", "pobre"]), validation["pobre"]
test_x, test_y = test.drop(columns=["id", "pobre"]), test["pobre"]

### Crear lista con 10 variables mas importantes del modelo grande (SHAP)
top_10_predictors = [
    "oc_household_average", "arriendo", "regimen_salud_1_household_average",
    "tipo_propiedad_vivienda_hogar", "ocupado_horas_trabajadas_normalmente_household_working",
    "numero_personas_unidad_gasto", "tipo_propiedad_vivienda_hogar_3",
    "ocupado_tamano_de_la_empresa_9_household_working", 
    "maximo_nivel_educativo_6_household_average", "ocupado_relab_4_household_working"
]

### 5.2 Realizar predicciones

In [ ]:
# Fijar variables
base_cols = ['id', 'pobre', 'hh_female', 'urbano', 'hh_informal', 'hh_oc']

# Predictions: Main Model
test_main = test[base_cols].copy()
test_main['predicted_prob'] = modelo.predict_proba(test_x)[:, 1]
test_main["predicted_class"] = np.where(test_main['predicted_prob'] >= 0.323,1,0)

# Predictions: Top-10 Model
test_top10 = test[base_cols].copy()
test_top10['predicted_prob'] = model_top_10.predict_proba(test_x[top_10_predictors])[:, 1]
test_top10["predicted_class"] = np.where(test_top10['predicted_prob'] >= 0.34,1,0)

# Crear indicadores de predicciones

for df in [test_main, test_top10]:
    df["tp"] = np.where(
        (df["pobre"] == 1) & (df["predicted_class"] == 1), 1, 0
    )

    df["tn"] = np.where(
        (df["pobre"] == 0) & (df["predicted_class"] == 0), 1, 0
    )

    df["fp"] = np.where(
        (df["pobre"] == 0) & (df["predicted_class"] == 1), 1, 0
    )

    df["fn"] = np.where(
        (df["pobre"] == 1) & (df["predicted_class"] == 0), 1, 0
    )

# limpiar la variable de informalidad

for df in[test_main, test_top10]:

    df["hh_informal"] = np.where(
        df["hh_oc"] == 1,
        df["hh_informal"],
        np.nan
    )

### 5.3 Mitigando la inequidad

#### 5.3.1 Modelo pequeño (Top 10)

In [ ]:
### Aplicar mitigacion
SEMILLA = 123
optimizador_top_10 = RelaxedThresholdOptimizer(predictor=lambda x: model_top_10.predict_proba(x)[:,1],
                                               constraint="equalized_odds",
                                               tolerance=0.0,
                                               seed = SEMILLA)

### Mitigación por residencia del hogar, urbano vs rural
optimizador_top_10.fit(X=validation_x[top_10_predictors],y=validation_y,group=validation_x["urbano"])
pred_relaxed_urbano_top_10 = optimizador_top_10(X=validation_x[top_10_predictors],group=validation_x["urbano"])
print(f"F1-score modelo (10), equalized odds :{f1_score(y_true=validation_y,y_pred=pred_relaxed_urbano_top_10):.3f}")

### Mitigación por status laboral del jefe del hogar
optimizador_top_10.fit(X=validation_x[top_10_predictors],y=validation_y,group=validation_x["hh_informal"])
pred_relaxed_informal_top_10 = optimizador_top_10(X=validation_x[top_10_predictors],group=validation_x["hh_informal"])
print(f"F1-score modelo (10), equalized odds :{f1_score(y_true=validation_y,y_pred=pred_relaxed_informal_top_10):.3f}")

#### 5.3.2 Definir función para computar métricas

In [ ]:
def calculate_cross_fairness_metrics(df_x, y_true, y_pred):
    """
    Evaluates overall performance and subgroup performance (F1, TPR, FPR)
    across all sensitive variables given an array of predictions.
    
    For 'hh_informal', evaluation is conditional on 'hh_oc == 1'.
    """
    sensitive_vars = ['urbano', 'hh_informal']
    results = []

    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)

    # --- A. Overall Performance ---
    tp_all = ((y_pred_arr == 1) & (y_true_arr == 1)).sum()
    fn_all = ((y_pred_arr == 0) & (y_true_arr == 1)).sum()
    fp_all = ((y_pred_arr == 1) & (y_true_arr == 0)).sum()
    tn_all = ((y_pred_arr == 0) & (y_true_arr == 0)).sum()

    tpr_all = tp_all / (tp_all + fn_all) if (tp_all + fn_all) > 0 else np.nan
    fpr_all = fp_all / (fp_all + tn_all) if (fp_all + tn_all) > 0 else np.nan
    f1_all = f1_score(y_true_arr, y_pred_arr, zero_division=0)

    results.append({
        'Variable Sensible': 'Rendimiento Agregado',
        'Grupo': 'Todos',
        'F1': f1_all,
        'TPR': tpr_all,
        'FPR': fpr_all
    })

    # --- B. Subgroup Performance ---
    for var in sensitive_vars:
        # Filter conditional on hh_oc == 1 for hh_informal
        if var == 'hh_informal' and 'hh_oc' in df_x.columns:
            mask_oc = (df_x['hh_oc'] == 1).values
        else:
            mask_oc = np.ones(len(df_x), dtype=bool)

        var_series = df_x[var].values

        for group_val in [1, 0]:
            mask_group = (var_series == group_val) & mask_oc
            
            y_t = y_true_arr[mask_group]
            y_p = y_pred_arr[mask_group]

            if len(y_t) == 0:
                continue

            tp = ((y_p == 1) & (y_t == 1)).sum()
            fn = ((y_p == 0) & (y_t == 1)).sum()
            fp = ((y_p == 1) & (y_t == 0)).sum()
            tn = ((y_p == 0) & (y_t == 0)).sum()

            tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
            f1 = f1_score(y_t, y_p, zero_division=0)

            group_label = 'Sí' if group_val == 1 else 'No'

            results.append({
                'Variable Sensible': var,
                'Grupo': group_label,
                'F1': f1,
                'TPR': tpr,
                'FPR': fpr
            })

    return pd.DataFrame(results)

#### 5.3.3 Aplicar función

In [ ]:
# Map generated predictions to their mitigation targets
predictions_dict = {
    'urbano': pred_relaxed_urbano_top_10,
    'hh_informal': pred_relaxed_informal_top_10
}

all_results = {}

for var_mitigated, y_pred in predictions_dict.items():
    metrics_df = calculate_cross_fairness_metrics(
        df_x=validation_x, 
        y_true=validation_y, 
        y_pred=y_pred
    )
    all_results[var_mitigated] = metrics_df

In [ ]:
formatted_dfs = []

for var_name, df_res in all_results.items():
    df_indexed = df_res.set_index(['Variable Sensible', 'Grupo'])
    df_indexed.columns = pd.MultiIndex.from_product([[f'Mitigado por: {var_name}'], df_indexed.columns])
    formatted_dfs.append(df_indexed)

# Combine into a side-by-side comparative table
final_fairness_table = pd.concat(formatted_dfs, axis=1)

# Display table formatted to 4 decimal places
final_fairness_table.round(4).reset_index()

### 5.4 Estilizar tabla para exportar

In [ ]:
def exportar_tabular_final(df, filename='../stores/output/05_mitigacion_justicia.tex'):
    # --- DICCIONARIOS DE MAPEO ---
    var_map = {
        "hh_female": "¿Jefe de hogar es de sexo femenino?",
        "urbano": "¿El hogar reside en una zona urbana?",
        "hh_informal": "¿El Jefe de hogar es un trabajador Informal?",
        "Rendimiento Agregado": "Rendimiento Agregado"
    }
    
    # Mapeo para los headers: quitamos el prefijo anterior
    header_map = {
        "hh_female": "¿Jefe de hogar es de sexo femenino?",
        "urbano": "¿El hogar reside en una zona urbana?",
        "hh_informal": "¿El Jefe de hogar es un trabajador Informal?"
    }
    
    group_map = {"Sí": "Sí", "No": "No", "Todos": "Todos", "í": "Sí", "o": "No"}

    df_temp = df.copy()
    if not isinstance(df_temp.index, pd.MultiIndex):
        df_temp = df_temp.set_index(['Variable Sensible', 'Grupo'])

    # Cálculo de estrellas
    metricas = ['F1', 'TPR', 'FPR']
    for col in df_temp.columns:
        if col[1] in metricas:
            diffs = df_temp[col].groupby(level=0).transform(lambda x: abs(x.max() - x.min()))
            def format_cell(val, diff):
                if diff == 0: return f"{val:.3f}***"
                if diff <= 0.03: return f"{val:.3f}**"
                if diff <= 0.05: return f"{val:.3f}*"
                return f"{val:.3f}"
            df_temp[col] = [format_cell(v, d) for v, d in zip(df_temp[col], diffs)]

    # --- CONSTRUCCIÓN LATEX ---
    latex_lines = [r"\begin{tabular}{llccccccccc}", r"\toprule"]
    
    # Encabezados: aplicamos el formato \makecell{Mitigado por : \\ Nombre}
    raw_headers = []
    for c in df_temp.columns:
        # Extraemos el nombre de la columna base limpiando "Mitigado por: "
        base_name = c[0].replace("Mitigado por:", "").strip()
        if base_name not in raw_headers:
            raw_headers.append(base_name)
    
    # Construcción de la línea de encabezados con makecell
    header_tex = " & & "
    parts = []
    for h in raw_headers:
        mapped_name = header_map.get(h, h)
        parts.append(f"\\multicolumn{{3}}{{c}}{{\\makecell{{Mitigado por : \\\\ {mapped_name}}}}}")
    
    latex_lines.append(" & & " + " & ".join(parts) + r" \\")
    latex_lines.append(r"\cmidrule(lr){3-5} \cmidrule(lr){6-8} \cmidrule(lr){9-11}")
    latex_lines.append("Variable Sensible & Grupo & " + " & ".join([c[1] for c in df_temp.columns]) + r" \\")
    latex_lines.append(r"\midrule")
    
    # Cuerpo con mapeo de variables y grupos
    unique_vars = df_temp.index.get_level_values(0).unique()
    for var in unique_vars:
        subset = df_temp.loc[var]
        var_label = var_map.get(str(var), str(var))
        
        if len(subset) >= 2:
            g1 = group_map.get(str(subset.index[0][1]), str(subset.index[0][1]))
            row1 = " & ".join([str(x) for x in subset.iloc[0].values])
            latex_lines.append(f"\\multirow{{2}}{{*}}{{{var_label}}} & {g1} & {row1} \\\\")
            
            g2 = group_map.get(str(subset.index[1][1]), str(subset.index[1][1]))
            row2 = " & ".join([str(x) for x in subset.iloc[1].values])
            latex_lines.append(f" & {g2} & {row2} \\\\")
        else:
            g1 = group_map.get(str(subset.index[0][1]), str(subset.index[0][1]))
            row = " & ".join([str(x) for x in subset.iloc[0].values])
            latex_lines.append(f"{var_label} & {g1} & {row} \\\\")
            
    latex_lines.extend([r"\bottomrule", r"\end{tabular}"])
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(latex_lines))
    
    return "\n".join(latex_lines)

# Ejecución
codigo_tex = exportar_tabular_final(final_fairness_table)

In [ ]:
# Open and read the .tex file
with open("../stores/output/05_mitigacion_justicia.tex", "r", encoding="utf-8") as file:
    latex_string = file.read()

latex_string = latex_string.replace("Rendimiento Agregado & No", "Rendimiento Agregado & ")

with open("../stores/output/05_mitigacion_justicia.tex", "w", encoding="utf-8") as file:
    file.write(latex_string)

## 6. Pruebas de robustez

In [ ]:
clean_globals()

In [ ]:
### Main Modules
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import fastparquet

### Models
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

### Metrics and calibrarion
from sklearn.metrics import (f1_score,accuracy_score,recall_score,precision_score,roc_auc_score,roc_curve,precision_recall_curve,confusion_matrix)

### Font sizes
plt.rcParams.update({
    'font.size': 14,          # Tamaño de fuente general
    'axes.labelsize': 16,     # Tamaño de las etiquetas de los ejes 
    'axes.titlesize': 16,     # Tamaño de los títulos de las facetas
    'xtick.labelsize': 12,    # Tamaño de los números en el eje X
    'ytick.labelsize': 12     # Tamaño de los números en el eje Y
})

### 6.1 Cargar datos y modelos

In [ ]:
### Cargar datos
train = pd.read_csv('../stores/input/01_original_data_train.csv')
test = pd.read_csv('../stores/input/01_original_data_test.csv')
validation = pd.read_csv('../stores/input/01_original_data_validation.csv')

### Limpiar dependiente
for df in [train,test,validation]:
    df["pobre"] = np.where(df["pobre"] == "Yes", 1, 0) 

### Split datos
train_x, train_y = train.drop(columns=["id","pobre"]), train["pobre"]
validation_x, validation_y = validation.drop(columns=["id","pobre"]), validation["pobre"]
test_x, test_y = test.drop(columns=["id","pobre"]), test["pobre"]

### Crear lista con 10 variables mas importantes del modelo grande (SHAP)
top_10_predictors = [
    "oc_household_average", "arriendo", "regimen_salud_1_household_average",
    "tipo_propiedad_vivienda_hogar", "ocupado_horas_trabajadas_normalmente_household_working",
    "numero_personas_unidad_gasto", "tipo_propiedad_vivienda_hogar_3",
    "ocupado_tamano_de_la_empresa_9_household_working", 
    "maximo_nivel_educativo_6_household_average", "ocupado_relab_4_household_working"
]

### Test ids
test_ids = test['id']

### 6.2 Hacer predicciones

#### 6.2.1 Crear semillas para reproducibilidad

In [ ]:
# 1. Generar 5000 numeros, con distribucion normal. Convertir a enteros y remover duplicados
unique_seeds = np.unique(np.abs(np.random.normal(loc=50000, scale=15000, size=5000)).astype(int))

# Mantener los ultimos 100 numeros generados y convertirlos a lista
seeds = unique_seeds[:100].tolist()

#### 6.2.2 Experimento de robustez para el modelo restringido (10 Variables)

In [ ]:
print("Esta parte del código es demorada, por eso se comenta y se carga su resultado")
print("Dado el caso de querer ejecturarlo, por favor descomentar el código")
# ### 1. Semillas para reproducibilidad
# seeds
# outputs = []

# for seed in seeds:
#     # 1. Bootstrap de los datos de entrenamiento 
#     train_x_boot = train_x.sample(frac=1.0, replace=True, random_state=seed)
#     train_y_boot = train_y.loc[train_x_boot.index]

#     # 2. Arquitectura del modelo
#     model = XGBClassifier(
#         n_estimators=500,
#         max_depth=7,
#         learning_rate=0.04,
#         min_child_weight=25,
#         gamma=0,
#         colsample_bytree=0.6,
#         subsample=0.8,
#         random_state=seed,
#         eval_metric='logloss')

#     # 3. Entrenar modelo
#     model.fit(X=train_x_boot[top_10_predictors], y=train_y_boot)

#     # 4. Computar predicciones en el conjunto de validacion
#     val_probs = model.predict_proba(validation_x[top_10_predictors])[:, 1]

#     # --- Calcular puntos de corte ---
#     precision, recall, pr_thresholds = precision_recall_curve(validation_y, val_probs)

#     # Calcular el F1 para cada punto de corte
#     with np.errstate(divide='ignore', invalid='ignore'):
#         val_f1_scores = np.nan_to_num((2 * precision * recall) / (precision + recall))

#     # Hallar el indice con el mayor F1
#     best_f1_idx = np.argmax(val_f1_scores)

#     # Computar el punto de corte
#     rfThreshF1 = pr_thresholds[best_f1_idx] if best_f1_idx < len(pr_thresholds) else 1.0

#     # 4. Predecir en el conjunto de entrenamietno usando el punto de corte optimizado para F1
#     test_probs = model.predict_proba(test_x[top_10_predictors])[:, 1]
#     test_preds_binary = (test_probs >= rfThreshF1).astype(int)

#     # Gaurdar resultados en un dataframe 
#     predicted_value_test = pd.DataFrame({
#         'seed': seed,                 # <--- ADDED: Track which seed this came from
#         'prob_Yes': test_probs,
#         'predicted_class': test_preds_binary,
#         'actual_pobre': test_y,
#     })
    
#     # --- Guardar el resultado
#     outputs.append(predicted_value_test)
    
# # --- Concatenar ---

# # Combine all predictions from all seeds into a single DataFrame
# all_predictions_top_10 = pd.concat(outputs, ignore_index=True)

# # Export the master prediction dataset to a CSV file
# all_predictions_top_10.to_parquet('../stores/output/07_sub_model_uncertainty_predicted_values.parquet',engine="fastparquet")

### 6.2.3 Experimento de robustez para el modelo completo (152 Variables)

In [ ]:
print("Esta parte del código es demorada, por eso se comenta y se carga su resultado")
print("Dado el caso de querer ejecturarlo, por favor descomentar el código")
# ### 1. Semillas para reproducibilidad
# seeds
# outputs = []

# for seed in seeds:
#     # 1. Bootstrap de los datos de entrenamiento 
#     train_x_boot = train_x.sample(frac=1.0, replace=True, random_state=seed)
#     train_y_boot = train_y.loc[train_x_boot.index]

#     # 2. Arquitectura del modelo
#     model = XGBClassifier(
#         n_estimators=500,
#         max_depth=7,
#         learning_rate=0.04,
#         min_child_weight=25,
#         gamma=0,
#         colsample_bytree=0.6,
#         subsample=0.8,
#         random_state=seed,
#         eval_metric='logloss')

#     # 3. Entrenar modelo
#     model.fit(X=train_x_boot, y=train_y_boot)

#     # 4. Computar predicciones en el conjunto de validacion
#     val_probs = model.predict_proba(validation_x)[:, 1]

#     # --- Calcular puntos de corte ---
#     precision, recall, pr_thresholds = precision_recall_curve(validation_y, val_probs)

#     # Calcular el F1 para cada punto de corte
#     with np.errstate(divide='ignore', invalid='ignore'):
#         val_f1_scores = np.nan_to_num((2 * precision * recall) / (precision + recall))

#     # Hallar el indice con el mayor F1
#     best_f1_idx = np.argmax(val_f1_scores)

#     # Computar el punto de corte
#     rfThreshF1 = pr_thresholds[best_f1_idx] if best_f1_idx < len(pr_thresholds) else 1.0

#     # 4. Predecir en el conjunto de entrenamietno usando el punto de corte optimizado para F1
#     test_probs = model.predict_proba(test_x)[:, 1]
#     test_preds_binary = (test_probs >= rfThreshF1).astype(int)

#     # Gaurdar resultados en un dataframe 
#     predicted_value_test = pd.DataFrame({
#         'seed': seed,                 # <--- ADDED: Track which seed this came from
#         'prob_Yes': test_probs,
#         'predicted_class': test_preds_binary,
#         'actual_pobre': test_y,
#     })
    
#     # --- Guardar el resultado
#     outputs.append(predicted_value_test)
    
# # --- Concatenar ---

# # Combine all predictions from all seeds into a single DataFrame
# all_predictions_main_model = pd.concat(outputs, ignore_index=True)

# # Export the master prediction dataset to a CSV file
# all_predictions_main_model.to_parquet('../stores/output/08_model_uncertainty_predicted_values.parquet',engine="fastparquet")

## 6.3 Computar métricas

In [ ]:
### importar datos
all_predictions_top_10 = pd.read_parquet('../stores/output/06_sub_model_uncertainty_predicted_values.parquet',engine="fastparquet")
all_predictions_main_model = pd.read_parquet('../stores/output/07_model_uncertainty_predicted_values.parquet',engine='fastparquet')


In [ ]:
# Helper function to compute all metrics for a single seed group
def compute_metrics(group):
    # scikit-learn metrics expect numeric formats for binary classification
    y_true = group['actual_pobre'].astype(int)
    y_pred = group['predicted_class'].astype(int)
    y_prob = group['prob_Yes']
    
    return pd.Series({
        'f1': f1_score(y_true, y_pred),
    })

#### 6.3.1 Modelo Principal (P = 152)

In [ ]:
### Computar estadísticas y unir datos

# Aplicar por semilla by seed, computar las metricas, and reset index
metrics_wide_main_model = all_predictions_main_model.groupby('seed').apply(compute_metrics).reset_index()

# Pivotear el dataframe
data_main_model = metrics_wide_main_model.melt(id_vars='seed', var_name='.metric', value_name='.estimate')

In [ ]:
# Calcular media y desviación estándar para cada metrica
summary_stats = data_main_model.groupby('.metric')['.estimate'].agg(
    mean_val='mean', 
    val_lower_bound=lambda x: x.quantile(0.025),
    val_upper_bound=lambda x: x.quantile(0.975)
).reset_index()

# Crear base del gráfico
g = sns.FacetGrid(data_main_model, col='.metric', col_wrap=3, sharex=False, height=4, aspect=1.2)

g.set_titles(col_template="{col_name}")

# Mapear el histograma
g.map_dataframe(sns.histplot, x='.estimate', fill=True, color='steelblue', 
                edgecolor='darkblue', alpha=0.5, bins=30)

# Iterar por cada panel del gráfico 
for ax, metric_name in zip(g.axes.flat, g.col_names):
    # Usar estadísticas precaculculadas
    stats = summary_stats[summary_stats['.metric'] == metric_name].iloc[0]
    mean_val = stats['mean_val']
    val_lower_bound = stats['val_lower_bound']
    val_upper_bound = stats['val_upper_bound']

    # Linea roja solida para le media
    ax.axvline(mean_val, color='red', linestyle='solid', linewidth=1.3)
    
    # Linea roja punteada para la media +/- desviacion estándar
    ax.axvline(val_lower_bound, color='red', linestyle='dashed', linewidth=1)
    ax.axvline(val_upper_bound, color='red', linestyle='dashed', linewidth=1)
    
    # Clean up axis labels
    ax.set_xlabel(" ")
    ax.set_ylabel("Conteo")

# Apply minimal theme layout equivalent
sns.set_theme(style="whitegrid")
plt.tight_layout()
plt.savefig("../stores/output/08_uncertainty_main_model.pdf", dpi=300, bbox_inches="tight")

#### 6.3.2 Modelo Restringido (P = 10)

In [ ]:
### Computar estadísticas y unir datos

# Aplicar por semilla by seed, computar las metricas, and reset index
metrics_wide_top_10 = all_predictions_top_10.groupby('seed').apply(compute_metrics).reset_index()

# Pivotear el dataframe
data_top_10 = metrics_wide_top_10.melt(id_vars='seed', var_name='.metric', value_name='.estimate')

In [ ]:
# Calcular media y desviación estándar para cada metrica
summary_stats_top_10 = data_top_10.groupby('.metric')['.estimate'].agg(
    mean_val='mean', 
    val_lower_bound=lambda x: x.quantile(0.025),
    val_upper_bound=lambda x: x.quantile(0.975)
).reset_index()

# Crear base del gráfico
g = sns.FacetGrid(data_top_10, col='.metric', col_wrap=3, sharex=False, height=4, aspect=1.2)

g.set_titles(col_template="{col_name}")

# Mapear el histograma
g.map_dataframe(sns.histplot, x='.estimate', fill=True, color='steelblue', 
                edgecolor='darkblue', alpha=0.5, bins=30)

# Iterar por cada panel del gráfico 
for ax, metric_name in zip(g.axes.flat, g.col_names):
    # Usar estadísticas precaculculadas
    stats = summary_stats_top_10[summary_stats['.metric'] == metric_name].iloc[0]
    mean_val = stats['mean_val']
    val_lower_bound = stats['val_lower_bound']
    val_upper_bound = stats['val_upper_bound']

    # Linea roja solida para le media
    ax.axvline(mean_val, color='red', linestyle='solid', linewidth=1.3)
    
    # Linea roja punteada para la media +/- desviacion estándar
    ax.axvline(val_lower_bound, color='red', linestyle='dashed', linewidth=1)
    ax.axvline(val_upper_bound, color='red', linestyle='dashed', linewidth=1)
    
    # Clean up axis labels
    ax.set_xlabel(" ")
    ax.set_ylabel("Conteo")

# Apply minimal theme layout equivalent
sns.set_theme(style="whitegrid")
plt.tight_layout()
plt.savefig("../stores/output/09_uncertainty_top_10_model.pdf", dpi=300, bbox_inches="tight")
plt.show()

## 7. Predicción conformal

In [ ]:
clean_globals()

In [ ]:
### Main Modules
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Models
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

### Metrics and calibrarion
from sklearn.metrics import (f1_score,accuracy_score,recall_score,precision_score,roc_auc_score,roc_curve,precision_recall_curve,confusion_matrix)

### Import conformal prediction wrapper
from crepes import WrapClassifier

### 7.1 Importar datos y modelo

In [ ]:
### Cargar modelo original
modelo_original = XGBClassifier()
modelo_original.load_model('../stores/input/01_original_r_trained_model_xgboost.ubj')

### Cargar modelo reducido (10 variables)
modelo_top_10 = XGBClassifier()
modelo_top_10.load_model('../stores/output/03_top_10.ubj')

### Cargar datos
train = pd.read_csv('../stores/input/01_original_data_train.csv')
test = pd.read_csv('../stores/input/01_original_data_test.csv')
validation = pd.read_csv('../stores/input/01_original_data_validation.csv')

### Limpiar dependiente
for df in [train,test,validation]:
    df["pobre"] = np.where(df["pobre"] == "Yes", 1, 0) 

### Split datos
train_x, train_y = train.drop(columns=["id","pobre"]), train["pobre"]
validation_x, validation_y = validation.drop(columns=["id","pobre"]), validation["pobre"]
test_x, test_y = test.drop(columns=["id","pobre"]), test["pobre"]

### Crear lista con 10 variables mas importantes del modelo grande (SHAP)
top_10_predictors = [
    "oc_household_average", "arriendo", "regimen_salud_1_household_average",
    "tipo_propiedad_vivienda_hogar", "ocupado_horas_trabajadas_normalmente_household_working",
    "numero_personas_unidad_gasto", "tipo_propiedad_vivienda_hogar_3",
    "ocupado_tamano_de_la_empresa_9_household_working", 
    "maximo_nivel_educativo_6_household_average", "ocupado_relab_4_household_working"
]

### 7.2 Crear el clasificador conformal

In [ ]:
## Setup modelo estándar (152 variables)
model = WrapClassifier(XGBClassifier(
                        n_estimators=500,
                        max_depth=7,
                        learning_rate=0.04,
                        min_child_weight=25,
                        gamma=0,
                        colsample_bytree=0.6,
                        subsample=0.8,
                        random_state=123,
                        eval_metric='logloss'))

## Setup modelo restringido (10 Variables)
model_top_10 = WrapClassifier(XGBClassifier(
                              n_estimators=500,
                              max_depth=7,
                              learning_rate=0.04,
                              min_child_weight=25,
                              gamma=0,
                              colsample_bytree=0.6,
                              subsample=0.8,
                              random_state=123,
                              eval_metric='logloss'))

### Ajustar modelos con los datos de entrenamiento
model.fit(train_x, train_y)
model_top_10.fit(train_x[top_10_predictors],train_y)

### Calibrar modelo con los datos de validación
model.calibrate(validation_x, validation_y)
model_top_10.calibrate(validation_x[top_10_predictors], validation_y)

### Evaluar modelos para un alfa de 0.05
print(model.evaluate(test_x, test_y, confidence=0.95))
print(model_top_10.evaluate(test_x[top_10_predictors], test_y, confidence=0.95))

### 7.3 Hacer predicciones

In [ ]:
### Predicciones del modelo base
predictions = test[["pobre"]]
predictions_top_10_model = predictions.assign(prediction_top_10_model = (model_top_10.predict_proba(test_x[top_10_predictors])[:,1] >= 0.323).astype(int))
prediction_main_model = predictions.assign(prediction_main_model = (model.predict_proba(test_x)[:,1] >= 0.34).astype(int))

### Realizar predicciones con el modelo calibrado
pred_sets_main_model = model.predict_set(test_x, confidence=0.95)
pred_sets_top_10_model = model_top_10.predict_set(test_x[top_10_predictors], confidence=0.95)

In [ ]:
### Marcar los conjuntos conformales
def define_conformal_set(df,conformal_set):

    ### Create boolean flags for each class
    df['contains_class_0'] = [0 in p_set for p_set in conformal_set]
    df['contains_class_1'] = [1 in p_set for p_set in conformal_set]

    ### Computing set size 
    df['set_size'] = [len(p_set) for p_set in conformal_set]

    ### Create indicator of prediction set

    # Conditions list
    condition_list = [
        (df["contains_class_0"] == True) & (df["contains_class_1"] == False),
        (df["contains_class_0"] == False) & (df["contains_class_1"] == True),
        (df["set_size"] == 2)
    ]

    # Choice list
    choice_list = ["0", "1", "(0,1)"]

    # Pass through conditional
    df["conformal_set"] = np.select(condition_list, choice_list, default="empty")

    return df

### Apply function
conformal_set_main_model = define_conformal_set(df=prediction_main_model,conformal_set=pred_sets_main_model).assign(model='Principal (p = 152)')
conformal_set_top_10_model = define_conformal_set(df=predictions_top_10_model,conformal_set=pred_sets_top_10_model).assign(model='Restringido (p = 10)')

### Bind together
predictions = pd.concat([conformal_set_main_model,conformal_set_top_10_model])

### 7.4 Construir tabla de predicciones

In [ ]:
### Make table
tab_count = predictions.groupby(["pobre","model"]).conformal_set.value_counts().reset_index()
tab_share = predictions.groupby(["pobre","model"]).conformal_set.value_counts(normalize=True).reset_index()
tab = pd.merge(left=tab_count,right=tab_share,how='inner',on=["model","pobre","conformal_set"])

### Reorder table
tab.sort_values(["model",'pobre','conformal_set'],ascending=True,inplace=True)

### Change scale of variable
tab["proportion"] = tab["proportion"] * 100

### Change 
tab["pobre"] = tab["pobre"].case_when([(tab["pobre"] == 0,'No Pobre'),
                                       (tab["pobre"] == 1,'Pobre')])

### Rename table
tab = (tab.
       rename(columns = {'pobre':"Etiqueta real",
                         'model':'Modelo',
                         "conformal_set":"Conjunto Conformal",
                         "count":"Conteo",
                         "proportion":"Proporción"}).
       reindex(columns=["Modelo","Etiqueta real","Conjunto Conformal","Conteo","Proporción"]).
       round(2))

### Set the columns as a MultiIndex
tab = tab.set_index(['Modelo', 'Etiqueta real'])

### 2. Sort the index
tab = tab.sort_index()

### 7.5 Guardar tabla

In [ ]:
tab.to_latex('../stores/output/10_conformal_prediction.tex',float_format="{:.2f}".format,escape=True,index=True,multirow=True)